# STEMNIST 完整压力帧脉冲编码

本 Notebook 严格复用 `python_solve_spike_encoding.ipynb` 的电路状态方程和脉冲检测参数，并用 Numba 对传感点并行编码。

它会处理 `data/pressure` 中的全部7700个样本：train 5390、val 1155、test 1155。每个样本的16×16传感点分别编码，最终输出保存到 `data/spike`，验证通过后自动生成 `data/spike.zip`。

## Linux环境准备

建议创建独立环境，避免NumPy与Numba版本冲突：

```bash
conda create -n stemnist-spike python=3.10 numpy=1.24 numba=0.58.1 ipykernel -y
conda activate stemnist-spike
python -m ipykernel install --user \
  --name stemnist-spike --display-name "Python (STEMNIST Spike)"
```

随后在Jupyter中选择 **Python (STEMNIST Spike)** 内核。

默认从当前目录及其父目录中自动寻找包含 `data/pressure/metadata.json` 的项目根目录。如果Jupyter从项目外启动，可在启动前设置：

```bash
export STEMNIST_PROJECT_ROOT=/path/to/STEMNIST_Classify
export STEMNIST_NUM_THREADS=32       # 可选，默认使用Numba可用线程数
export STEMNIST_BATCH_SIZE=8         # 可选，默认4个样本一批
```

输出结构：

```text
data/spike.zip            # 可提交到GitHub的压缩包
data/spike/
├── metadata.json
├── train/
│   ├── spike.npy
│   └── manifest.csv
├── val/
│   ├── spike.npy
│   └── manifest.csv
└── test/
    ├── spike.npy
    └── manifest.csv
```

编码过程支持批次级断点续算，未完成数据暂存在 `data/.spike-building`。

In [1]:
# 1. 导入依赖并设置跨平台路径
from __future__ import annotations

import csv
import json
import math
import os
import shutil
import time
from collections import Counter
from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path

import numpy as np

try:
    from tqdm.auto import tqdm
except ImportError as error:
    raise RuntimeError(
        "缺少进度条依赖。请先执行：pip install -r requirements.txt"
    ) from error

try:
    import numba
    from numba import get_num_threads, njit, prange, set_num_threads
except ImportError as error:
    raise RuntimeError(
        "Numba无法导入。请按上方说明创建并选择兼容的Jupyter内核。"
    ) from error


def find_project_root():
    """通过环境变量或当前目录向上寻找项目根目录。"""
    override = os.environ.get("STEMNIST_PROJECT_ROOT")
    start = Path(override).expanduser() if override else Path.cwd()
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "data" / "pressure" / "metadata.json").is_file():
            return candidate

    raise FileNotFoundError(
        "未找到项目根目录。请在项目目录中启动Jupyter，或设置"
        "STEMNIST_PROJECT_ROOT。"
    )


def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"环境变量{name}必须是布尔值，实际为{value!r}")


PROJECT_ROOT = find_project_root()
PRESSURE_ROOT = PROJECT_ROOT / "data" / "pressure"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "spike"
BUILD_ROOT = PROJECT_ROOT / "data" / ".spike-building"
ARCHIVE_PATH = PROJECT_ROOT / "data" / "spike.zip"

SAMPLE_BATCH_SIZE = max(
    1,
    int(os.environ.get("STEMNIST_BATCH_SIZE", "4")),
)
available_threads = get_num_threads()
requested_threads = int(
    os.environ.get("STEMNIST_NUM_THREADS", str(available_threads))
)
NUM_THREADS = max(1, min(requested_threads, available_threads))
OVERWRITE = environment_flag("STEMNIST_OVERWRITE", default=False)
ARCHIVE_OVERWRITE = environment_flag(
    "STEMNIST_ARCHIVE_OVERWRITE",
    default=False,
)

LABELS = tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ123456789")
SPLITS = ("train", "val", "test")
FRAME_COUNT = 240
HEIGHT = 16
WIDTH = 16
EXPECTED_SOURCE_COUNTS = {"train": 5390, "val": 1155, "test": 1155}
EXPECTED_PER_CLASS = {"train": 154, "val": 33, "test": 33}

set_num_threads(NUM_THREADS)
print(f"项目目录：{PROJECT_ROOT}")
print(f"输出目录：{OUTPUT_ROOT}")
print(f"Numba线程：{get_num_threads()}")
print(f"样本批大小：{SAMPLE_BATCH_SIZE}")
print(f"NumPy：{np.__version__}")
print(f"Numba：{numba.__version__}")

项目目录：/root/autodl-tmp/STEMNIST_Classify
输出目录：/root/autodl-tmp/STEMNIST_Classify/data/spike
Numba线程：208
样本批大小：4
NumPy：2.3.2
Numba：0.67.0


/root/miniconda3/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [2]:
# 2. 读取完整train/val/test清单，并保持与pressure.npy逐行对齐
OUTPUT_MANIFEST_FIELDS = (
    "row_index",
    "source_row_index",
    "sample_id",
    "participant_id",
    "label",
    "label_index",
    "repetition",
    "source_member",
    "source_split",
)


def read_csv_rows(path):
    with Path(path).open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def normalize_source_row(row, split_name):
    source_row_index = int(row["row_index"])
    return {
        "row_index": source_row_index,
        "source_row_index": source_row_index,
        "sample_id": row["sample_id"],
        "participant_id": row["participant_id"],
        "label": row["label"],
        "label_index": int(row["label_index"]),
        "repetition": int(row["repetition"]),
        "source_member": row.get("source_member", ""),
        "source_split": split_name,
    }


def load_source_records(pressure_root=PRESSURE_ROOT):
    records_by_split = {}
    all_sample_ids = []

    for split_name in SPLITS:
        pressure_path = pressure_root / split_name / "pressure.npy"
        manifest_path = pressure_root / split_name / "manifest.csv"
        if not pressure_path.is_file() or not manifest_path.is_file():
            raise FileNotFoundError(f"缺少pressure输入：{pressure_root / split_name}")

        pressure = np.load(pressure_path, mmap_mode="r", allow_pickle=False)
        expected_shape = (
            EXPECTED_SOURCE_COUNTS[split_name],
            FRAME_COUNT,
            HEIGHT,
            WIDTH,
        )
        if pressure.shape != expected_shape or pressure.dtype != np.uint8:
            raise RuntimeError(
                f"{split_name} pressure格式错误："
                f"shape={pressure.shape}, dtype={pressure.dtype}"
            )
        del pressure

        records = [
            normalize_source_row(row, split_name)
            for row in read_csv_rows(manifest_path)
        ]
        records.sort(key=lambda row: row["source_row_index"])

        if len(records) != EXPECTED_SOURCE_COUNTS[split_name]:
            raise RuntimeError(f"{split_name} manifest行数不正确")
        if [row["source_row_index"] for row in records] != list(
            range(len(records))
        ):
            raise RuntimeError(f"{split_name} manifest行号不连续")

        class_counts = Counter(row["label"] for row in records)
        if set(class_counts) != set(LABELS):
            raise RuntimeError(f"{split_name}类别集合不完整")
        if set(class_counts.values()) != {EXPECTED_PER_CLASS[split_name]}:
            raise RuntimeError(
                f"{split_name}每类样本数错误：{dict(class_counts)}"
            )

        records_by_split[split_name] = records
        all_sample_ids.extend(row["sample_id"] for row in records)

    if len(all_sample_ids) != 7700 or len(set(all_sample_ids)) != 7700:
        raise RuntimeError("完整数据集应包含7700个互不重复的样本")

    return records_by_split


SOURCE_RECORDS = load_source_records()
for split_name in SPLITS:
    print(f"{split_name}: {len(SOURCE_RECORDS[split_name])}个样本")

train: 5390个样本
val: 1155个样本
test: 1155个样本


In [3]:
# 3. 参考电路模型：保留原始Python语义，用于一致性验证
@dataclass(frozen=True)
class SystemModelParameters:
    duration: float = 2.0
    dt: float = 20e-6

    vin_u1: float = 2.5
    vh: float = 1.676
    vl: float = 1.127
    rin_nbox: float = 89213.44
    rme_nbox: float = 806.0
    cparal: float = 1e-6
    u2_rload: float = 10e3

    synapse_vth: float = 0.679973290013
    synapse_vnorm: float = 2.0
    synapse_pdrive: float = 1.1281797291
    ron1: float = 651321.797132
    roff1: float = 58783.4118027
    ron2: float = 6453.3547737
    roff2: float = 7225.92691131
    ron3: float = 54391.4170977
    roff3: float = 22092.0764053
    g11: float = 0.000663574393119
    g12: float = 0.00122698089109
    g13: float = 4.42225017007e-05
    g23: float = 0.000671314326731
    gv: float = 6.561726644e-06
    synapse_vds: float = 1.0

    rf1: float = 2.5e3
    cf1: float = 10e-9
    vref: float = 0.75
    rin2: float = 10e3
    rf2: float = 10e3
    negative_rail: float = -5.0
    positive_rail: float = 5.0


PARAMETERS = SystemModelParameters()
V_ARM = 1.55
V_FIRE = 1.25
REFRACTORY = 0.5e-3
BIN_COUNT = FRAME_COUNT
STEP_COUNT = int(round(PARAMETERS.duration / PARAMETERS.dt))
TIME_GRID = np.linspace(
    0.0,
    PARAMETERS.duration,
    STEP_COUNT + 1,
    dtype=np.float64,
)
ACTUAL_DT = float(TIME_GRID[1] - TIME_GRID[0])
BIN_DT = PARAMETERS.duration / BIN_COUNT


def build_pressure_pwl_arrays(pressure_sequence, edge_time=1e-9):
    """在内存中构造与参考Notebook相同的PWL控制点。"""
    pressure_sequence = np.asarray(pressure_sequence, dtype=np.float64)
    if pressure_sequence.shape != (FRAME_COUNT,):
        raise ValueError(f"压力序列形状必须是 ({FRAME_COUNT},)")

    pressure_sequence = np.clip(pressure_sequence, 0.0, 255.0)
    frame_dt = PARAMETERS.duration / FRAME_COUNT
    edge_time = min(edge_time, frame_dt / 1000.0)
    times = [0.0]
    values = [pressure_sequence[0]]

    for frame_index, pressure_value in enumerate(pressure_sequence):
        frame_end = (frame_index + 1) * frame_dt
        times.append(frame_end - edge_time)
        values.append(pressure_value)
        if frame_index + 1 < FRAME_COUNT:
            times.append(frame_end)
            values.append(pressure_sequence[frame_index + 1])

    times.append(PARAMETERS.duration)
    values.append(pressure_sequence[-1])
    return np.asarray(times), np.asarray(values)


def sample_pressure_on_grid(pressure_sequence):
    times, values = build_pressure_pwl_arrays(pressure_sequence)
    return np.interp(TIME_GRID, times, values)


# 每个20微秒仿真点对应的原始压力帧。若存在过渡区采样点则立即报错。
symbolic_grid = sample_pressure_on_grid(np.arange(FRAME_COUNT))
if not np.allclose(symbolic_grid, np.rint(symbolic_grid), atol=1e-12):
    raise RuntimeError("仿真时间点落入PWL的1ns过渡区，不能使用帧索引优化")
FRAME_INDEX_BY_STEP = np.rint(symbolic_grid).astype(np.int16)


def advance_nbox_python(
    voltage,
    high_resistance,
    input_voltage,
    load_resistance,
    dt,
):
    nbox_resistance = (
        PARAMETERS.rin_nbox if high_resistance else PARAMETERS.rme_nbox
    )
    target = (
        input_voltage
        * nbox_resistance
        / (load_resistance + nbox_resistance)
    )
    tau = PARAMETERS.cparal / (
        1.0 / load_resistance + 1.0 / nbox_resistance
    )
    next_voltage = target + (voltage - target) * math.exp(-dt / tau)

    crossed_high = high_resistance and next_voltage >= PARAMETERS.vh
    crossed_low = (not high_resistance) and next_voltage <= PARAMETERS.vl
    if not (crossed_high or crossed_low):
        return next_voltage, high_resistance

    threshold = PARAMETERS.vh if high_resistance else PARAMETERS.vl
    ratio = (threshold - target) / (voltage - target)
    crossing_time = -tau * math.log(ratio)
    remaining_time = max(dt - crossing_time, 0.0)
    high_resistance = not high_resistance

    nbox_resistance = (
        PARAMETERS.rin_nbox if high_resistance else PARAMETERS.rme_nbox
    )
    target = (
        input_voltage
        * nbox_resistance
        / (load_resistance + nbox_resistance)
    )
    tau = PARAMETERS.cparal / (
        1.0 / load_resistance + 1.0 / nbox_resistance
    )
    next_voltage = target + (threshold - target) * math.exp(
        -remaining_time / tau
    )
    return next_voltage, high_resistance


def simulate_reference_sequence(pressure_sequence):
    """运行一条传感点序列，返回参考final_out。"""
    pressure_grid = sample_pressure_on_grid(pressure_sequence)
    final_out = np.zeros(STEP_COUNT + 1, dtype=np.float64)

    u1_high = True
    u2_high = True
    u1_value = 0.0
    vtia_value = 0.0
    state_x1 = 0.0
    state_x2 = 0.0
    state_x3 = 0.0

    x1_on_decay = math.exp(-ACTUAL_DT / (PARAMETERS.ron1 * 1e-6))
    x1_off_decay = math.exp(-ACTUAL_DT / (PARAMETERS.roff1 * 1e-6))
    x2_on_decay = math.exp(-ACTUAL_DT / (PARAMETERS.ron2 * 1e-6))
    x2_off_decay = math.exp(-ACTUAL_DT / (PARAMETERS.roff2 * 1e-6))
    x3_on_decay = math.exp(-ACTUAL_DT / (PARAMETERS.ron3 * 1e-6))
    x3_off_decay = math.exp(-ACTUAL_DT / (PARAMETERS.roff3 * 1e-6))
    tia_decay = math.exp(-ACTUAL_DT / (PARAMETERS.rf1 * PARAMETERS.cf1))

    for index in range(1, STEP_COUNT + 1):
        pressure = pressure_grid[index - 1]
        if pressure > 75.0:
            u1_rload = np.clip(
                900e3 / max(pressure - 75.0, 1e-6),
                5e3,
                1e12,
            )
        else:
            u1_rload = 1e12

        u1_value, u1_high = advance_nbox_python(
            u1_value,
            u1_high,
            PARAMETERS.vin_u1,
            u1_rload,
            ACTUAL_DT,
        )
        normalized_gate = max(
            (u1_value - PARAMETERS.synapse_vth)
            / (PARAMETERS.synapse_vnorm - PARAMETERS.synapse_vth),
            0.0,
        ) ** PARAMETERS.synapse_pdrive

        if u1_value > PARAMETERS.synapse_vth:
            state_x1 = normalized_gate + (
                state_x1 - normalized_gate
            ) * x1_on_decay
            state_x2 = normalized_gate + (
                state_x2 - normalized_gate
            ) * x2_on_decay
            state_x3 = normalized_gate + (
                state_x3 - normalized_gate
            ) * x3_on_decay
        else:
            state_x1 *= x1_off_decay
            state_x2 *= x2_off_decay
            state_x3 *= x3_off_decay

        conductance = max(
            PARAMETERS.g11 * state_x1 * state_x1
            + PARAMETERS.g12 * state_x1 * state_x2
            + PARAMETERS.g13 * state_x1 * state_x3
            + PARAMETERS.g23 * state_x2 * state_x3
            + PARAMETERS.gv * normalized_gate,
            0.0,
        )
        current = PARAMETERS.synapse_vds * conductance
        tia_target = -PARAMETERS.rf1 * current
        vtia_value = tia_target + (vtia_value - tia_target) * tia_decay
        vtia_value = min(
            max(vtia_value, PARAMETERS.negative_rail),
            PARAMETERS.positive_rail,
        )

        gain = PARAMETERS.rf2 / PARAMETERS.rin2
        drive = (1.0 + gain) * PARAMETERS.vref - gain * vtia_value
        drive = min(
            max(drive, PARAMETERS.negative_rail),
            PARAMETERS.positive_rail,
        )

        final_out[index], u2_high = advance_nbox_python(
            final_out[index - 1],
            u2_high,
            drive,
            PARAMETERS.u2_rload,
            ACTUAL_DT,
        )

    return final_out


def detect_and_bin_reference(final_out):
    """使用参考阈值和不应期，输出240步二值脉冲。"""
    armed = False
    last_spike_time = -np.inf
    spikes = np.zeros(BIN_COUNT, dtype=np.uint8)

    for index in range(1, len(TIME_GRID)):
        if final_out[index] >= V_ARM:
            armed = True

        falling_crossing = (
            final_out[index - 1] > V_FIRE
            and final_out[index] <= V_FIRE
        )
        outside_refractory = (
            TIME_GRID[index] - last_spike_time >= REFRACTORY
        )

        if armed and falling_crossing and outside_refractory:
            spike_time = TIME_GRID[index]
            last_spike_time = spike_time
            armed = False
            if 0.0 <= spike_time < PARAMETERS.duration:
                bin_index = int(np.floor(spike_time / BIN_DT))
                spikes[bin_index] = 1

    return spikes


def reference_encode_sequence(pressure_sequence):
    return detect_and_bin_reference(
        simulate_reference_sequence(pressure_sequence)
    )

In [4]:
# 4. Numba并行编码：模拟、检测和分箱融合在一个内核中
# Numba不能直接读取Python dataclass，因此将同一组固定参数映射为标量常量。
N_DURATION = PARAMETERS.duration
N_VIN_U1 = PARAMETERS.vin_u1
N_VH = PARAMETERS.vh
N_VL = PARAMETERS.vl
N_RIN_NBOX = PARAMETERS.rin_nbox
N_RME_NBOX = PARAMETERS.rme_nbox
N_CPARAL = PARAMETERS.cparal
N_U2_RLOAD = PARAMETERS.u2_rload
N_SYNAPSE_VTH = PARAMETERS.synapse_vth
N_SYNAPSE_VNORM = PARAMETERS.synapse_vnorm
N_SYNAPSE_PDRIVE = PARAMETERS.synapse_pdrive
N_RON1 = PARAMETERS.ron1
N_ROFF1 = PARAMETERS.roff1
N_RON2 = PARAMETERS.ron2
N_ROFF2 = PARAMETERS.roff2
N_RON3 = PARAMETERS.ron3
N_ROFF3 = PARAMETERS.roff3
N_G11 = PARAMETERS.g11
N_G12 = PARAMETERS.g12
N_G13 = PARAMETERS.g13
N_G23 = PARAMETERS.g23
N_GV = PARAMETERS.gv
N_SYNAPSE_VDS = PARAMETERS.synapse_vds
N_RF1 = PARAMETERS.rf1
N_CF1 = PARAMETERS.cf1
N_VREF = PARAMETERS.vref
N_RIN2 = PARAMETERS.rin2
N_RF2 = PARAMETERS.rf2
N_NEGATIVE_RAIL = PARAMETERS.negative_rail
N_POSITIVE_RAIL = PARAMETERS.positive_rail


@njit(inline="always", fastmath=False)
def advance_nbox_numba(
    voltage,
    high_resistance,
    input_voltage,
    load_resistance,
    dt,
):
    nbox_resistance = (
        N_RIN_NBOX if high_resistance else N_RME_NBOX
    )
    target = (
        input_voltage
        * nbox_resistance
        / (load_resistance + nbox_resistance)
    )
    tau = N_CPARAL / (
        1.0 / load_resistance + 1.0 / nbox_resistance
    )
    next_voltage = target + (voltage - target) * math.exp(-dt / tau)

    crossed_high = high_resistance and next_voltage >= N_VH
    crossed_low = (not high_resistance) and next_voltage <= N_VL
    if not (crossed_high or crossed_low):
        return next_voltage, high_resistance

    threshold = N_VH if high_resistance else N_VL
    ratio = (threshold - target) / (voltage - target)
    crossing_time = -tau * math.log(ratio)
    remaining_time = max(dt - crossing_time, 0.0)
    high_resistance = not high_resistance

    nbox_resistance = (
        N_RIN_NBOX if high_resistance else N_RME_NBOX
    )
    target = (
        input_voltage
        * nbox_resistance
        / (load_resistance + nbox_resistance)
    )
    tau = N_CPARAL / (
        1.0 / load_resistance + 1.0 / nbox_resistance
    )
    next_voltage = target + (threshold - target) * math.exp(
        -remaining_time / tau
    )
    return next_voltage, high_resistance


@njit(parallel=True, fastmath=False)
def encode_taxel_sequences_numba(
    pressure_sequences,
    frame_index_by_step,
    time_grid,
):
    """将[M,240]压力序列编码为[M,240]二值脉冲。"""
    sequence_count = pressure_sequences.shape[0]
    encoded = np.zeros((sequence_count, BIN_COUNT), dtype=np.uint8)

    for sequence_index in prange(sequence_count):
        u1_high = True
        u2_high = True
        u1_value = 0.0
        vtia_value = 0.0
        final_value = 0.0
        state_x1 = 0.0
        state_x2 = 0.0
        state_x3 = 0.0
        armed = False
        last_spike_time = -np.inf

        x1_on_decay = math.exp(-ACTUAL_DT / (N_RON1 * 1e-6))
        x1_off_decay = math.exp(-ACTUAL_DT / (N_ROFF1 * 1e-6))
        x2_on_decay = math.exp(-ACTUAL_DT / (N_RON2 * 1e-6))
        x2_off_decay = math.exp(-ACTUAL_DT / (N_ROFF2 * 1e-6))
        x3_on_decay = math.exp(-ACTUAL_DT / (N_RON3 * 1e-6))
        x3_off_decay = math.exp(-ACTUAL_DT / (N_ROFF3 * 1e-6))
        tia_decay = math.exp(-ACTUAL_DT / (N_RF1 * N_CF1))

        for step_index in range(1, STEP_COUNT + 1):
            frame_index = frame_index_by_step[step_index - 1]
            pressure = float(pressure_sequences[sequence_index, frame_index])

            if pressure > 75.0:
                u1_rload = 900e3 / max(pressure - 75.0, 1e-6)
                u1_rload = min(max(u1_rload, 5e3), 1e12)
            else:
                u1_rload = 1e12

            u1_value, u1_high = advance_nbox_numba(
                u1_value,
                u1_high,
                N_VIN_U1,
                u1_rload,
                ACTUAL_DT,
            )
            normalized_gate = max(
                (u1_value - N_SYNAPSE_VTH)
                / (N_SYNAPSE_VNORM - N_SYNAPSE_VTH),
                0.0,
            ) ** N_SYNAPSE_PDRIVE

            if u1_value > N_SYNAPSE_VTH:
                state_x1 = normalized_gate + (
                    state_x1 - normalized_gate
                ) * x1_on_decay
                state_x2 = normalized_gate + (
                    state_x2 - normalized_gate
                ) * x2_on_decay
                state_x3 = normalized_gate + (
                    state_x3 - normalized_gate
                ) * x3_on_decay
            else:
                state_x1 *= x1_off_decay
                state_x2 *= x2_off_decay
                state_x3 *= x3_off_decay

            conductance = (
                N_G11 * state_x1 * state_x1
                + N_G12 * state_x1 * state_x2
                + N_G13 * state_x1 * state_x3
                + N_G23 * state_x2 * state_x3
                + N_GV * normalized_gate
            )
            conductance = max(conductance, 0.0)
            current = N_SYNAPSE_VDS * conductance

            tia_target = -N_RF1 * current
            vtia_value = tia_target + (vtia_value - tia_target) * tia_decay
            vtia_value = min(
                max(vtia_value, N_NEGATIVE_RAIL),
                N_POSITIVE_RAIL,
            )
            gain = N_RF2 / N_RIN2
            drive = (1.0 + gain) * N_VREF - gain * vtia_value
            drive = min(
                max(drive, N_NEGATIVE_RAIL),
                N_POSITIVE_RAIL,
            )

            previous_final = final_value
            final_value, u2_high = advance_nbox_numba(
                final_value,
                u2_high,
                drive,
                N_U2_RLOAD,
                ACTUAL_DT,
            )

            if final_value >= V_ARM:
                armed = True

            current_time = time_grid[step_index]
            falling_crossing = (
                previous_final > V_FIRE and final_value <= V_FIRE
            )
            outside_refractory = (
                current_time - last_spike_time >= REFRACTORY
            )
            if armed and falling_crossing and outside_refractory:
                last_spike_time = current_time
                armed = False
                if current_time < N_DURATION:
                    bin_index = int(math.floor(current_time / BIN_DT))
                    if 0 <= bin_index < BIN_COUNT:
                        encoded[sequence_index, bin_index] = 1

    return encoded


def encode_pressure_batch(pressure_batch):
    """将[B,240,16,16]转换为相同形状的二值脉冲。"""
    pressure_batch = np.asarray(pressure_batch)
    if pressure_batch.ndim != 4 or pressure_batch.shape[1:] != (
        FRAME_COUNT,
        HEIGHT,
        WIDTH,
    ):
        raise ValueError(f"pressure_batch形状错误：{pressure_batch.shape}")
    if pressure_batch.dtype != np.uint8:
        raise TypeError(f"pressure_batch必须为uint8：{pressure_batch.dtype}")

    batch_size = pressure_batch.shape[0]
    sequences = np.ascontiguousarray(
        pressure_batch.transpose(0, 2, 3, 1).reshape(-1, FRAME_COUNT)
    )
    encoded_sequences = encode_taxel_sequences_numba(
        sequences,
        FRAME_INDEX_BY_STEP,
        TIME_GRID,
    )
    return np.ascontiguousarray(
        encoded_sequences
        .reshape(batch_size, HEIGHT, WIDTH, FRAME_COUNT)
        .transpose(0, 3, 1, 2)
    )

In [5]:
# 5. 一致性测试与完整数据集耗时估计
train_records = SOURCE_RECORDS["train"]
at_record = next(row for row in train_records if row["sample_id"] == "AT_A_1")
train_pressure = np.load(
    PRESSURE_ROOT / "train" / "pressure.npy",
    mmap_mode="r",
    allow_pickle=False,
)
at_pressure = np.asarray(
    train_pressure[at_record["source_row_index"]]
).copy()
del train_pressure

# 先用一条序列触发JIT编译，编译时间不计入后续基准。
warmup_sequence = np.ascontiguousarray(at_pressure[:, 0, 0][None, :])
encode_taxel_sequences_numba(
    warmup_sequence,
    FRAME_INDEX_BY_STEP,
    TIME_GRID,
)

test_coordinates = ((9, 6), (0, 0), (8, 8), (15, 15))
test_sequences = np.ascontiguousarray(
    np.stack([at_pressure[:, row, col] for row, col in test_coordinates])
)
numba_spikes = encode_taxel_sequences_numba(
    test_sequences,
    FRAME_INDEX_BY_STEP,
    TIME_GRID,
)

for index, (row, col) in enumerate(test_coordinates):
    reference_spikes = reference_encode_sequence(at_pressure[:, row, col])
    if not np.array_equal(reference_spikes, numba_spikes[index]):
        mismatch = np.flatnonzero(reference_spikes != numba_spikes[index])
        raise AssertionError(
            f"AT_A_1 ({row},{col}) 的Numba结果与参考实现不一致：{mismatch}"
        )

expected_reference_bins = np.array([201, 203, 204, 206, 208])
actual_reference_bins = np.flatnonzero(numba_spikes[0])
if not np.array_equal(actual_reference_bins, expected_reference_bins):
    raise AssertionError(
        "AT_A_1 (9,6) 未复现参考结果："
        f"期望={expected_reference_bins}, 实际={actual_reference_bins}"
    )

benchmark_start = time.perf_counter()
at_encoded = encode_pressure_batch(at_pressure[None, ...])
seconds_per_sample = time.perf_counter() - benchmark_start
total_samples = sum(EXPECTED_SOURCE_COUNTS.values())
estimated_seconds = seconds_per_sample * total_samples

if not np.array_equal(at_encoded[0, :, 9, 6], numba_spikes[0]):
    raise AssertionError("整样本编码与单传感点编码不一致")

print("参考实现与Numba实现逐元素一致。")
print(f"AT_A_1 (9,6) 激活分箱：{actual_reference_bins.tolist()}")
print(f"单样本256个传感点耗时：{seconds_per_sample:.3f}秒")
print(f"完整7700个样本预计耗时：{estimated_seconds / 60.0:.2f}分钟")

参考实现与Numba实现逐元素一致。
AT_A_1 (9,6) 激活分箱：[201, 203, 204, 206, 208]
单样本256个传感点耗时：0.322秒
完整7700个样本预计耗时：41.29分钟


In [6]:
# 6. 可恢复的完整数据集导出
ENCODER_VERSION = "strict-python-circuit-numba-full-v2"


def sha256_file(path, block_size=1024 * 1024):
    digest = sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def atomic_write_json(path, value):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(value, file, ensure_ascii=False, indent=2)
    temporary_path.replace(path)


def close_memmap(array):
    array.flush()
    memory_map = getattr(array, "_mmap", None)
    if memory_map is not None:
        memory_map.close()


def validate_safe_output_path(path):
    path = Path(path).resolve()
    data_root = (PROJECT_ROOT / "data").resolve()
    if path == data_root or data_root not in path.parents:
        raise ValueError(f"输出路径必须位于项目data目录中：{path}")


def build_configuration_payload():
    source_metadata = read_json(PRESSURE_ROOT / "metadata.json")
    manifest_hashes = {
        split_name: sha256_file(
            PRESSURE_ROOT / split_name / "manifest.csv"
        )
        for split_name in SPLITS
    }
    return {
        "encoder_version": ENCODER_VERSION,
        "dataset_scope": "full",
        "model_parameters": asdict(PARAMETERS),
        "spike_detection": {
            "v_arm": V_ARM,
            "v_fire": V_FIRE,
            "refractory": REFRACTORY,
            "bin_count": BIN_COUNT,
            "binary": True,
        },
        "source_counts": EXPECTED_SOURCE_COUNTS,
        "source_zip_md5": source_metadata["source_zip_md5"],
        "source_manifest_sha256": manifest_hashes,
        "numpy_version": np.__version__,
        "numba_version": numba.__version__,
    }


def configuration_fingerprint(payload):
    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return sha256(serialized).hexdigest()


def write_output_manifest(path, records):
    with Path(path).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=OUTPUT_MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(records)


def initialize_or_resume_build(overwrite=False):
    validate_safe_output_path(OUTPUT_ROOT)
    validate_safe_output_path(BUILD_ROOT)

    payload = build_configuration_payload()
    fingerprint = configuration_fingerprint(payload)

    if overwrite:
        if OUTPUT_ROOT.exists():
            shutil.rmtree(OUTPUT_ROOT)
        if BUILD_ROOT.exists():
            shutil.rmtree(BUILD_ROOT)
    elif OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError(
            f"完整输出已经存在：{OUTPUT_ROOT}。"
            "如需重建，请设置STEMNIST_OVERWRITE=1。"
        )

    progress_path = BUILD_ROOT / "progress.json"
    if BUILD_ROOT.exists():
        if not progress_path.is_file():
            raise RuntimeError(
                f"发现无法恢复的临时目录：{BUILD_ROOT}。"
                "请检查后设置STEMNIST_OVERWRITE=1重建。"
            )
        progress = read_json(progress_path)
        if progress.get("fingerprint") != fingerprint:
            raise RuntimeError(
                "临时结果的参数、环境或输入与当前配置不同，拒绝错误续算。"
                "请设置STEMNIST_OVERWRITE=1重建。"
            )
        print(f"继续已有进度：{progress['completed_rows']}")
        return progress, payload, fingerprint

    BUILD_ROOT.mkdir(parents=True, exist_ok=False)
    for split_name in SPLITS:
        split_dir = BUILD_ROOT / split_name
        split_dir.mkdir()
        records = SOURCE_RECORDS[split_name]
        write_output_manifest(split_dir / "manifest.csv", records)

        spike_array = np.lib.format.open_memmap(
            split_dir / "spike.npy",
            mode="w+",
            dtype=np.uint8,
            shape=(len(records), FRAME_COUNT, HEIGHT, WIDTH),
        )
        spike_array[:] = 0
        close_memmap(spike_array)

    progress = {
        "fingerprint": fingerprint,
        "completed_rows": {split_name: 0 for split_name in SPLITS},
    }
    atomic_write_json(progress_path, progress)
    return progress, payload, fingerprint


def validate_build_outputs():
    all_sample_ids = []
    for split_name in SPLITS:
        records = SOURCE_RECORDS[split_name]
        spike = np.load(
            BUILD_ROOT / split_name / "spike.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        expected_shape = (
            EXPECTED_SOURCE_COUNTS[split_name],
            FRAME_COUNT,
            HEIGHT,
            WIDTH,
        )
        if spike.shape != expected_shape or spike.dtype != np.uint8:
            raise RuntimeError(
                f"{split_name}输出格式错误：{spike.shape}, {spike.dtype}"
            )
        if not np.all((spike == 0) | (spike == 1)):
            raise RuntimeError(f"{split_name}包含非二值脉冲")

        manifest_rows = read_csv_rows(BUILD_ROOT / split_name / "manifest.csv")
        if len(manifest_rows) != EXPECTED_SOURCE_COUNTS[split_name]:
            raise RuntimeError(f"{split_name}输出manifest行数错误")
        if [int(row["row_index"]) for row in manifest_rows] != list(
            range(EXPECTED_SOURCE_COUNTS[split_name])
        ):
            raise RuntimeError(f"{split_name}输出manifest行号错误")

        class_counts = Counter(row["label"] for row in manifest_rows)
        if set(class_counts.values()) != {EXPECTED_PER_CLASS[split_name]}:
            raise RuntimeError(f"{split_name}输出类别数量错误")

        all_sample_ids.extend(row["sample_id"] for row in records)
        del spike

    if len(all_sample_ids) != 7700 or len(set(all_sample_ids)) != 7700:
        raise RuntimeError("输出样本ID不完整或存在重复")


def encode_full_dataset(overwrite=OVERWRITE):
    progress, payload, fingerprint = initialize_or_resume_build(overwrite)
    progress_path = BUILD_ROOT / "progress.json"
    started_at = time.perf_counter()

    completed_rows = {
        split_name: int(progress["completed_rows"][split_name])
        for split_name in SPLITS
    }
    for split_name in SPLITS:
        row_count = len(SOURCE_RECORDS[split_name])
        if not 0 <= completed_rows[split_name] <= row_count:
            raise RuntimeError(
                f"{split_name}进度越界：{completed_rows[split_name]}"
            )

    total_samples = sum(len(SOURCE_RECORDS[name]) for name in SPLITS)
    completed_samples = sum(completed_rows.values())

    # 进度条以已持久化的样本数为准，断点续算时从已有位置继续。
    with tqdm(
        total=total_samples,
        initial=completed_samples,
        desc="脉冲编码",
        unit="样本",
        dynamic_ncols=True,
        mininterval=0.5,
        smoothing=0.1,
        leave=True,
    ) as progress_bar:
        for split_name in SPLITS:
            records = SOURCE_RECORDS[split_name]
            start_row = completed_rows[split_name]
            progress_bar.set_postfix(
                split=split_name,
                split_progress=f"{start_row}/{len(records)}",
                refresh=True,
            )

            # 当前split已经完成时不再打开对应的内存映射文件。
            if start_row == len(records):
                continue

            source_pressure = np.load(
                PRESSURE_ROOT / split_name / "pressure.npy",
                mmap_mode="r",
                allow_pickle=False,
            )
            output_spike = np.load(
                BUILD_ROOT / split_name / "spike.npy",
                mmap_mode="r+",
                allow_pickle=False,
            )

            try:
                for batch_start in range(
                    start_row,
                    len(records),
                    SAMPLE_BATCH_SIZE,
                ):
                    batch_end = min(
                        batch_start + SAMPLE_BATCH_SIZE,
                        len(records),
                    )
                    # 保持与pressure数组的样本顺序严格一致。
                    pressure_batch = np.asarray(
                        source_pressure[batch_start:batch_end]
                    ).copy()
                    encoded_batch = encode_pressure_batch(pressure_batch)
                    output_spike[batch_start:batch_end] = encoded_batch
                    output_spike.flush()

                    # 先保存断点，再更新显示，确保进度条代表可恢复进度。
                    progress["completed_rows"][split_name] = batch_end
                    atomic_write_json(progress_path, progress)
                    progress_bar.update(batch_end - batch_start)
                    progress_bar.set_postfix(
                        split=split_name,
                        split_progress=f"{batch_end}/{len(records)}",
                        refresh=False,
                    )
            finally:
                close_memmap(output_spike)
                del source_pressure

    print("编码完成，正在验证输出……")
    validate_build_outputs()
    elapsed_seconds = time.perf_counter() - started_at
    metadata = dict(payload)
    metadata.update(
        {
            "fingerprint": fingerprint,
            "complete": True,
            "counts": EXPECTED_SOURCE_COUNTS,
            "output_shapes": {
                split_name: [
                    EXPECTED_SOURCE_COUNTS[split_name],
                    FRAME_COUNT,
                    HEIGHT,
                    WIDTH,
                ]
                for split_name in SPLITS
            },
            "output_dtype": "uint8",
            "elapsed_seconds_this_run": elapsed_seconds,
        }
    )
    atomic_write_json(BUILD_ROOT / "metadata.json", metadata)
    progress_path.unlink()

    if OUTPUT_ROOT.exists():
        if any(OUTPUT_ROOT.iterdir()):
            raise RuntimeError(f"最终输出目录意外为非空：{OUTPUT_ROOT}")
        OUTPUT_ROOT.rmdir()
    BUILD_ROOT.replace(OUTPUT_ROOT)

    print(f"完整脉冲数据已生成：{OUTPUT_ROOT}")
    return metadata


In [ ]:
# 7. 执行完整7700个样本的编码
ENCODING_METADATA = encode_full_dataset(overwrite=OVERWRITE)
ENCODING_METADATA

脉冲编码:   0%|          | 0/7700 [00:00<?, ?样本/s]

In [ ]:
# 8. 验证最终输出
all_sample_ids = []
for split_name in SPLITS:
    spike = np.load(
        OUTPUT_ROOT / split_name / "spike.npy",
        mmap_mode="r",
        allow_pickle=False,
    )
    rows = read_csv_rows(OUTPUT_ROOT / split_name / "manifest.csv")
    class_counts = Counter(row["label"] for row in rows)

    expected_shape = (
        EXPECTED_SOURCE_COUNTS[split_name],
        FRAME_COUNT,
        HEIGHT,
        WIDTH,
    )
    assert spike.shape == expected_shape
    assert spike.dtype == np.uint8
    assert np.all((spike == 0) | (spike == 1))
    assert len(rows) == EXPECTED_SOURCE_COUNTS[split_name]
    assert [int(row["row_index"]) for row in rows] == list(
        range(EXPECTED_SOURCE_COUNTS[split_name])
    )
    assert set(class_counts) == set(LABELS)
    assert set(class_counts.values()) == {EXPECTED_PER_CLASS[split_name]}

    active_ratio = float(np.count_nonzero(spike)) / spike.size
    print(
        f"{split_name}: shape={spike.shape}, "
        f"active_ratio={active_ratio:.6%}"
    )
    all_sample_ids.extend(row["sample_id"] for row in rows)
    del spike

assert len(all_sample_ids) == 7700
assert len(set(all_sample_ids)) == 7700

train_rows = read_csv_rows(OUTPUT_ROOT / "train" / "manifest.csv")
at_row = next(row for row in train_rows if row["sample_id"] == "AT_A_1")
train_spike = np.load(
    OUTPUT_ROOT / "train" / "spike.npy",
    mmap_mode="r",
    allow_pickle=False,
)
assert np.array_equal(
    np.flatnonzero(train_spike[int(at_row["row_index"]), :, 9, 6]),
    np.array([201, 203, 204, 206, 208]),
)
del train_spike

metadata = read_json(OUTPUT_ROOT / "metadata.json")
assert metadata["complete"] is True
assert metadata["dataset_scope"] == "full"
print("完整数据集全部验证通过。")

In [ ]:
# 9. 将完整脉冲数据压缩为 data/spike.zip
from zipfile import ZIP_DEFLATED, ZipFile


ARCHIVE_REQUIRED_FILES = {
    "spike/metadata.json",
    "spike/train/spike.npy",
    "spike/train/manifest.csv",
    "spike/val/spike.npy",
    "spike/val/manifest.csv",
    "spike/test/spike.npy",
    "spike/test/manifest.csv",
}


def create_spike_archive(
    source_dir=OUTPUT_ROOT,
    archive_path=ARCHIVE_PATH,
    overwrite=ARCHIVE_OVERWRITE,
):
    """以最高DEFLATE等级压缩7个完整输出文件，并原子写入ZIP。"""
    source_dir = Path(source_dir).resolve()
    archive_path = Path(archive_path).resolve()
    validate_safe_output_path(source_dir)

    metadata_path = source_dir / "metadata.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"缺少编码元数据：{metadata_path}")
    metadata = read_json(metadata_path)
    if metadata.get("complete") is not True:
        raise RuntimeError("脉冲数据尚未完整生成，拒绝压缩")
    if metadata.get("dataset_scope") != "full":
        raise RuntimeError("当前输出不是完整数据集，拒绝压缩")

    source_files = sorted(
        (source_dir / relative.removeprefix("spike/"))
        for relative in ARCHIVE_REQUIRED_FILES
    )
    missing = [path for path in source_files if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"缺少待压缩文件：{missing}")

    if archive_path.exists() and not overwrite:
        raise FileExistsError(
            f"压缩包已经存在：{archive_path}。"
            "如需重建，请设置STEMNIST_ARCHIVE_OVERWRITE=1。"
        )

    temporary_path = archive_path.with_name(f".{archive_path.name}.tmp")
    temporary_path.unlink(missing_ok=True)

    try:
        with ZipFile(
            temporary_path,
            mode="w",
            compression=ZIP_DEFLATED,
            compresslevel=9,
            allowZip64=True,
        ) as archive:
            for path in source_files:
                archive_name = path.relative_to(source_dir.parent).as_posix()
                archive.write(
                    path,
                    arcname=archive_name,
                    compress_type=ZIP_DEFLATED,
                    compresslevel=9,
                )

        with ZipFile(temporary_path, "r") as archive:
            member_names = {
                info.filename for info in archive.infolist() if not info.is_dir()
            }
            if member_names != ARCHIVE_REQUIRED_FILES:
                raise RuntimeError(
                    "ZIP成员不正确："
                    f"缺失={sorted(ARCHIVE_REQUIRED_FILES - member_names)}，"
                    f"额外={sorted(member_names - ARCHIVE_REQUIRED_FILES)}"
                )
            bad_member = archive.testzip()
            if bad_member is not None:
                raise RuntimeError(f"ZIP CRC校验失败：{bad_member}")

        temporary_path.replace(archive_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    source_bytes = sum(path.stat().st_size for path in source_files)
    archive_bytes = archive_path.stat().st_size
    archive_sha256 = sha256_file(archive_path)

    print(f"压缩完成：{archive_path}")
    print(f"原始大小：{source_bytes / 1024**2:.2f} MiB")
    print(f"ZIP大小：{archive_bytes / 1024**2:.2f} MiB")
    print(f"压缩率：{archive_bytes / source_bytes:.4%}")
    print(f"SHA-256：{archive_sha256}")

    if archive_bytes > 100 * 1024**2:
        print("警告：文件超过GitHub普通Git单文件限制，请使用Git LFS。")
    elif archive_bytes > 25 * 1024**2:
        print("文件应通过Git命令行提交，不要使用GitHub网页上传。")
    else:
        print("文件大小适合通过普通Git或GitHub网页上传。")

    return {
        "path": str(archive_path),
        "size_bytes": archive_bytes,
        "sha256": archive_sha256,
        "members": sorted(ARCHIVE_REQUIRED_FILES),
    }


ARCHIVE_METADATA = create_spike_archive()
ARCHIVE_METADATA